
# SHA-256 Carry Exhaust, Glass Key, and Full Trace

This notebook builds a **fully working traced SHA-256** implementation and separates three objects that are often conflated:

1. **Digest / hash** — the standard 256-bit SHA-256 output.
2. **Z-axis Glass Key** — the wordwise difference between the final state of a real run and the final state of the **NOP backbone** (`W[t] = 0` for all 64 rounds).
3. **Full trace** — the per-round execution record: registers, message words, schedule words, and carry observables.

## Core point

These are **not the same thing**.

- The **hash** is the visible value channel.
- The **Z-axis Glass Key** is a structural perturbation measured against the NOP backbone.
- The **full trace** is the stronger object: it contains the actual round-by-round execution history.

For a **single-block message**, the notebook also shows:

- the traced implementation matches `hashlib.sha256`,
- the NOP backbone is computable directly from `IV` and `K`,
- the Z-axis Glass Key is easy to compute,
- the original single-block message can be reconstructed from the **full trace** because `W[0..15]` are the padded message block words,
- but the **Z-axis Glass Key alone is not the same as the message**.


In [ ]:

# Setup / install cell
# No third-party packages are required for this notebook.
# Standard library only.


In [ ]:

import hashlib
import struct
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any

MASK32 = 0xFFFFFFFF

IV = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

def rotr(x: int, n: int) -> int:
    return ((x >> n) | (x << (32 - n))) & MASK32

def ch(x: int, y: int, z: int) -> int:
    return (x & y) ^ (~x & z)

def maj(x: int, y: int, z: int) -> int:
    return (x & y) ^ (x & z) ^ (y & z)

def big_sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def big_sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def small_sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def small_sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def add32(x: int, y: int) -> Tuple[int, int]:
    total = x + y
    return total & MASK32, 1 if total > MASK32 else 0

def add_many_stepwise(values: List[int]) -> Tuple[int, List[int]]:
    acc = values[0] & MASK32
    carries: List[int] = []
    for v in values[1:]:
        acc, carry = add32(acc, v & MASK32)
        carries.append(carry)
    return acc, carries

def sha256_pad(message: bytes) -> bytes:
    bit_len = len(message) * 8
    padded = message + b"\x80"
    while (len(padded) % 64) != 56:
        padded += b"\x00"
    padded += struct.pack(">Q", bit_len)
    return padded

def words_from_block(block: bytes) -> List[int]:
    return list(struct.unpack(">16I", block))

def unpad_sha256_block(block: bytes) -> bytes:
    if len(block) != 64:
        raise ValueError("Expected exactly one 64-byte block.")
    try:
        marker_idx = block.index(0x80)
    except ValueError as exc:
        raise ValueError("No 0x80 marker found in padded block.") from exc

    length_bytes = block[-8:]
    bit_len = struct.unpack(">Q", length_bytes)[0]
    msg_len = bit_len // 8
    if marker_idx != msg_len:
        raise ValueError("Padding marker does not agree with length field.")
    if any(b != 0 for b in block[msg_len + 1:-8]):
        raise ValueError("Non-zero bytes found inside padding zone.")
    return block[:msg_len]

def fmt32(x: int) -> str:
    return f"{x:08x}"

def fmt_state(state: List[int]) -> str:
    labels = "abcdefgh"
    return " ".join(f"{labels[i]}={state[i]:08x}" for i in range(8))


In [ ]:

@dataclass
class RoundTrace:
    round_idx: int
    state_in: List[int]
    state_out: List[int]
    w: int
    k: int
    S0: int
    S1: int
    Ch: int
    Maj: int
    T1: int
    T2: int
    t1_step_carries: List[int]   # four binary flags from the staged T1 accumulation
    t2_carry: int                # one carry from S0 + Maj
    carry_e: int                 # one carry from d + T1
    carry_a: int                 # one carry from T1 + T2

    def short_row(self) -> Dict[str, Any]:
        return {
            "r": self.round_idx,
            "W": fmt32(self.w),
            "K": fmt32(self.k),
            "T1": fmt32(self.T1),
            "T2": fmt32(self.T2),
            "t1_steps": "".join(str(b) for b in self.t1_step_carries),
            "t2": self.t2_carry,
            "e_out": self.carry_e,
            "a_out": self.carry_a,
            "a_in": fmt32(self.state_in[0]),
            "e_in": fmt32(self.state_in[4]),
            "a_out_word": fmt32(self.state_out[0]),
            "e_out_word": fmt32(self.state_out[4]),
        }

class TracedSHA256:
    def __init__(self, iv: List[int] = None, k: List[int] = None):
        self.iv = list(IV if iv is None else iv)
        self.k = list(K if k is None else k)

    def build_schedule(self, block_words: List[int]) -> List[int]:
        w = list(block_words) + [0] * 48
        for t in range(16, 64):
            s0 = small_sigma0(w[t - 15])
            s1 = small_sigma1(w[t - 2])
            w[t] = (w[t - 16] + s0 + w[t - 7] + s1) & MASK32
        return w

    def compress_schedule_with_trace(
        self,
        schedule: List[int],
        initial_state: List[int] = None,
        final_add: bool = True,
    ) -> Tuple[List[int], List[RoundTrace]]:
        state = list(self.iv if initial_state is None else initial_state)
        traces: List[RoundTrace] = []

        a, b, c, d, e, f, g, h = state
        for t in range(64):
            state_in = [a, b, c, d, e, f, g, h]

            S1 = big_sigma1(e)
            Ch = ch(e, f, g)
            T1, t1_step_carries = add_many_stepwise([h, S1, Ch, self.k[t], schedule[t]])
            S0 = big_sigma0(a)
            Maj = maj(a, b, c)
            T2, t2_carry = add32(S0, Maj)
            e_out, carry_e = add32(d, T1)
            a_out, carry_a = add32(T1, T2)

            h2 = g
            g2 = f
            f2 = e
            e2 = e_out
            d2 = c
            c2 = b
            b2 = a
            a2 = a_out
            state_out = [a2, b2, c2, d2, e2, f2, g2, h2]

            traces.append(
                RoundTrace(
                    round_idx=t,
                    state_in=state_in,
                    state_out=state_out,
                    w=schedule[t],
                    k=self.k[t],
                    S0=S0,
                    S1=S1,
                    Ch=Ch,
                    Maj=Maj,
                    T1=T1,
                    T2=T2,
                    t1_step_carries=t1_step_carries,
                    t2_carry=t2_carry,
                    carry_e=carry_e,
                    carry_a=carry_a,
                )
            )

            a, b, c, d, e, f, g, h = state_out

        working = [a, b, c, d, e, f, g, h]
        if final_add:
            working = [(state[i] + working[i]) & MASK32 for i in range(8)]
        return working, traces

    def hash_with_trace(self, message: bytes) -> Dict[str, Any]:
        padded = sha256_pad(message)
        blocks = [padded[i:i+64] for i in range(0, len(padded), 64)]

        h = list(self.iv)
        all_traces: List[List[RoundTrace]] = []
        all_block_words: List[List[int]] = []
        all_schedules: List[List[int]] = []

        for block in blocks:
            block_words = words_from_block(block)
            schedule = self.build_schedule(block_words)
            h, traces = self.compress_schedule_with_trace(schedule, initial_state=h, final_add=True)
            all_block_words.append(block_words)
            all_schedules.append(schedule)
            all_traces.append(traces)

        digest = "".join(f"{x:08x}" for x in h)
        return {
            "message": message,
            "padded": padded,
            "blocks": blocks,
            "block_words": all_block_words,
            "schedules": all_schedules,
            "final_state": h,
            "digest": digest,
            "traces": all_traces,
        }

    def run_nop_backbone(self, initial_state: List[int] = None, final_add: bool = True) -> Dict[str, Any]:
        zero_schedule = [0] * 64
        final_state, traces = self.compress_schedule_with_trace(
            zero_schedule,
            initial_state=self.iv if initial_state is None else initial_state,
            final_add=final_add,
        )
        return {
            "schedule": zero_schedule,
            "final_state": final_state,
            "traces": traces,
        }

def z_axis_glass_key(final_state: List[int], nop_final_state: List[int]) -> List[int]:
    return [(final_state[i] - nop_final_state[i]) & MASK32 for i in range(8)]

def render_round_rows(traces: List[RoundTrace], n: int = 8) -> List[Dict[str, Any]]:
    return [tr.short_row() for tr in traces[:n]]

def print_rows(rows: List[Dict[str, Any]]) -> None:
    if not rows:
        print("(no rows)")
        return
    headers = list(rows[0].keys())
    widths = {h: max(len(str(h)), *(len(str(r[h])) for r in rows)) for h in headers}
    header_line = " | ".join(f"{h:{widths[h]}}" for h in headers)
    sep_line = "-+-".join("-" * widths[h] for h in headers)
    print(header_line)
    print(sep_line)
    for row in rows:
        print(" | ".join(f"{str(row[h]):{widths[h]}}" for h in headers))

def summarize_carry_channel(traces: List[RoundTrace]) -> Dict[str, Any]:
    t1_step_flags = [bit for tr in traces for bit in tr.t1_step_carries]
    t2_flags = [tr.t2_carry for tr in traces]
    e_flags = [tr.carry_e for tr in traces]
    a_flags = [tr.carry_a for tr in traces]

    return {
        "rounds": len(traces),
        "t1_step_flags_total": len(t1_step_flags),
        "t1_step_flags_set": sum(t1_step_flags),
        "t2_flags_total": len(t2_flags),
        "t2_flags_set": sum(t2_flags),
        "e_output_flags_total": len(e_flags),
        "e_output_flags_set": sum(e_flags),
        "a_output_flags_total": len(a_flags),
        "a_output_flags_set": sum(a_flags),
        "instrumented_binary_carry_observables_total": len(t1_step_flags) + len(t2_flags) + len(e_flags) + len(a_flags),
    }

def single_block_recover_from_trace(run: Dict[str, Any]) -> bytes:
    if len(run["blocks"]) != 1:
        raise ValueError("This demo recovers from the trace only for single-block messages.")
    block_words = run["block_words"][0]
    block = struct.pack(">16I", *block_words)
    return unpad_sha256_block(block)



## Run a real traced example

Use a message short enough to stay inside one 512-bit block after SHA-256 padding.  
That lets us demonstrate direct reconstruction from the trace.


In [ ]:

engine = TracedSHA256()

message = b"hello"
run = engine.hash_with_trace(message)
nop = engine.run_nop_backbone()
glass_key_z = z_axis_glass_key(run["final_state"], nop["final_state"])

print("Message:", message)
print("Digest (traced) :", run["digest"])
print("Digest (hashlib):", hashlib.sha256(message).hexdigest())
print("Match:", run["digest"] == hashlib.sha256(message).hexdigest())
print()

print("Final state:")
print([fmt32(x) for x in run["final_state"]])
print("NOP final state:")
print([fmt32(x) for x in nop["final_state"]])
print("Z-axis Glass Key = final_state - nop_final_state (mod 2^32):")
print([fmt32(x) for x in glass_key_z])



## First rounds of the real trace

The table below shows the beginning of the execution history.

- `W` is the schedule word injected at that round.
- `t1_steps` is the sequence of **four staged carry flags** produced while building `T1 = h + Σ1(e) + Ch + K[t] + W[t]`.
- `t2` is the carry from `T2 = Σ0(a) + Maj(a,b,c)`.
- `e_out` is the carry from `d + T1`.
- `a_out` is the carry from `T1 + T2`.

This is a concrete, instrumented **carry-observable channel**.


In [ ]:

rows = render_round_rows(run["traces"][0], n=10)
print_rows(rows)



## Carry-channel summary

This notebook uses **explicit measured carry observables**.  
It does **not** assume an abstract “1,792 carry bits” reservoir.

Here the instrumented binary channel is:

- 4 staged `T1` carry flags per round,
- 1 `T2` carry flag per round,
- 1 output carry for `e = d + T1`,
- 1 output carry for `a = T1 + T2`.

That is **7 binary carry observables per round**, i.e. **448 per 64-round block** in this particular instrumented view.

If you only care about the staged `T1` chain, then the notebook exposes **256** binary flags per block.  
Different instrumentation choices produce different counts. That is exactly why the channel should be kept separate from broader “residue budget” narratives.


In [ ]:

carry_summary = summarize_carry_channel(run["traces"][0])
for k, v in carry_summary.items():
    print(f"{k}: {v}")



## Is the Glass Key “just the message”?

No.

This notebook makes the distinction explicit:

### A. Hash / digest
The standard 256-bit SHA-256 output.

### B. Z-axis Glass Key
The difference:
\[
\Delta_Z = x_{64}^{(\text{real})} - x_{64}^{(\text{NOP})} \pmod{2^{32}}
\]
This is a **structural footprint relative to the NOP backbone**.  
It is not just the original message bytes.

### C. Full trace
The round-by-round execution record.  
This is the stronger object. It includes the message schedule words and the carry observables.

For a single-block message, the full trace lets us recover the original padded block directly, because `W[0..15]` are the 16 original block words before schedule expansion.


In [ ]:

recovered = single_block_recover_from_trace(run)

print("Recovered from full trace:", recovered)
print("Matches original:", recovered == message)
print()

print("First 16 original block words (W[0..15]):")
print([fmt32(x) for x in run["block_words"][0]])
print()

print("Expanded words example (W[0..19]):")
print([fmt32(x) for x in run["schedules"][0][:20]])



## Why the Z-axis Glass Key is not the same as the full trace

The Z-axis Glass Key is only 8 words here — the final-state perturbation relative to the NOP backbone.

The full trace contains 64 rounds of state transitions and a full expanded schedule.  
So even before discussing invertibility, they are **different objects with different information content**.


In [ ]:

print("Digest size (bytes):", len(bytes.fromhex(run["digest"])))
print("Z-axis Glass Key size (bytes):", len(glass_key_z) * 4)
print("Rounds in full trace:", len(run["traces"][0]))
print("Original block words recorded:", len(run["block_words"][0]))
print("Expanded schedule words recorded:", len(run["schedules"][0]))



## NOP backbone check

The NOP backbone uses the same SHA-256 round machinery, but with:

\[
W[t] = 0 \quad \text{for all} \quad t = 0,\dots,63
\]

This is **not** the hash of the empty string.  
It is a message-free backbone used as a reference trajectory.


In [ ]:

nop_rows = render_round_rows(nop["traces"], n=8)
print_rows(nop_rows)

t2_0 = nop["traces"][0].T2
print()
print("NOP T2[0] =", fmt32(t2_0))



## Final takeaway

For this notebook, keep the objects separate:

- **hash** = the visible SHA-256 output,
- **carry exhaust / carry observables** = measured overflow-side residue from modular additions,
- **Z-axis Glass Key** = final-state perturbation relative to the NOP backbone,
- **full trace** = the actual round-by-round execution history.

So to your question:

> “Is this the same as the Glass Key data, or is Glass Key just the message?”

**Answer:** no. The Glass Key is **not just the message**.  
And the weaker **Z-axis Glass Key** is also **not the same thing** as the stronger **full trace**.

The notebook shows both, running end to end.
